In [1]:
!pip install -U transformers accelerate -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.7 MB/s eta 0:00:00


In [2]:
import json
import re
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

# ==========================================
# 1. LOAD MODEL & PROCESSOR GEMMA LOKAL
# ==========================================

MODEL_ID = "google/gemma-4-E4B-it"

print("Memuat processor dan model Gemma lokal...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

def generate_text(prompt: str, max_new_tokens: int = 512) -> str:
    """Fungsi helper untuk inferensi teks LLM langsung."""
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens, 
            temperature=0.1, 
            do_sample=False
        )
    # Dekode keluaran tanpa mengulang input prompt
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return processor.decode(generated_ids, skip_special_tokens=True).strip()


# ==========================================
# 2. DUMMY DATABASE & TOOLS MANUAL
# ==========================================

DUMMY_DATABASE = {
    "USR-001": {
        "name": "Budi Santoso",
        "recent_login_country": "Singapura",
        "daily_limit_idr": 50_000_000,
        "spent_today_idr": 75_000_000,
        "phone_number": "+6281234567890",
        "card_frozen": False
    }
}

def get_login_history(user_id: str) -> str:
    user = DUMMY_DATABASE.get(user_id)
    if not user:
        return "User tidak ditemukan."
    return f"Lokasi login terakhir nasabah {user_id} ({user['name']}): {user['recent_login_country']}."

def check_card_limit(user_id: str) -> str:
    user = DUMMY_DATABASE.get(user_id)
    if not user:
        return "User tidak ditemukan."
    return (
        f"Limit harian: IDR {user['daily_limit_idr']:,}. "
        f"Penggunaan hari ini: IDR {user['spent_today_idr']:,}."
    )

def freeze_card(user_id: str) -> str:
    user = DUMMY_DATABASE.get(user_id)
    if not user:
        return "User tidak ditemukan."
    user["card_frozen"] = True
    return f"SUKSES: Kartu untuk user {user_id} ({user['name']}) telah dibekukan sementara."

def send_fraud_alert_notification(user_id: str, message: str) -> str:
    user = DUMMY_DATABASE.get(user_id)
    if not user:
        return "User tidak ditemukan."
    return f"SUKSES: Notifikasi dikirim ke {user['phone_number']} dengan pesan: '{message}'"

# Pemetaan nama tool ke fungsi Python
TOOL_REGISTRY = {
    "get_login_history": get_login_history,
    "check_card_limit": check_card_limit,
    "freeze_card": freeze_card,
    "send_fraud_alert_notification": send_fraud_alert_notification
}

TOOL_DESCRIPTIONS = """
1. get_login_history(user_id): Mendapatkan riwayat lokasi login terakhir nasabah.
2. check_card_limit(user_id): Mengecek limit harian dan total transaksi hari ini.
3. freeze_card(user_id): Membekukan kartu nasabah sementara.
4. send_fraud_alert_notification(user_id, message): Mengirimkan pesan SMS/WhatsApp konfirmasi transaksi ke nasabah.
"""


# ==========================================
# 3. ROUTER MANUAL
# ==========================================

def router_node(trigger_text: str) -> str:
    prompt = f"""Anda adalah router pesan di sistem perbankan.
Klasifikasikan pesan berikut ke dalam salah satu kategori:
- AML_AGENT: Jika berisi peringatan kecurangan, anomali, atau transaksi mencurigakan.
- GENERAL_QUERY: Jika berisi pertanyaan umum perbankan.

Pesan: {trigger_text}

Respon HANYA dengan 'AML_AGENT' atau 'GENERAL_QUERY'."""

    response = generate_text(prompt, max_new_tokens=20)
    if "AML_AGENT" in response.upper():
        return "AML_AGENT"
    return "GENERAL_QUERY"


# ==========================================
# 4. REACT AGENT LOOP MANUAL
# ==========================================

def execute_tool(action_name: str, action_input: str) -> str:
    """Eksekutor tool dinamis berdasarkan parsing string."""
    func = TOOL_REGISTRY.get(action_name.strip())
    if not func:
        return f"Error: Tool '{action_name}' tidak ditemukan."
    
    # Bersihkan kutip jika argumen dibungkus string
    clean_input = action_input.strip().strip("'\"")
    
    try:
        # Jika fungsi membutuhkan 2 argumen (seperti send_fraud_alert_notification)
        if "," in clean_input:
            args = [arg.strip().strip("'\"") for arg in clean_input.split(",", 1)]
            return func(args[0], args[1])
        return func(clean_input)
    except Exception as e:
        return f"Error eksekusi tool: {str(e)}"


def run_react_agent(query: str, max_turns: int = 5) -> str:
    prompt_header = f"""Anda adalah Asisten Analis Investigasi AML (Anti-Money Laundering) Bank.
Tugas Anda adalah menganalisis anomali transaksi nasabah menggunakan tools yang ada, lalu mengambil tindakan yang tepat.

Tools yang tersedia:
{TOOL_DESCRIPTIONS}

Aturan Format Komunikasi (Wajib diikuti ketat):
Question: [Input pertanyaan/masalah]
Thought: [Langkah pemikiran Anda]
Action: [Nama tool yang ingin digunakan dari daftar]
Action Input: [Input untuk tool tersebut]
Observation: [Hasil respon dari tool]
... (Ulangi Thought/Action/Action Input/Observation jika masih butuh informasi)
Thought: [Pemikiran akhir setelah analisis cukup]
Final Answer: [Ringkasan hasil investigasi dan tindakan akhir]

Question: {query}
"""
    scratchpad = ""

    for turn in range(max_turns):
        full_prompt = prompt_header + scratchpad + "\nThought:"
        raw_response = generate_text(full_prompt, max_new_tokens=256)
        
        # Tambahkan kembali 'Thought:' yang terpotong saat eksekusi
        step_text = "Thought:" + raw_response
        print(f"\n--- [TURN {turn + 1}] ---")
        print(step_text)

        # Cek jika agen sudah mencapai Final Answer
        if "Final Answer:" in step_text:
            final_answer = step_text.split("Final Answer:")[1].strip()
            return final_answer

        # Parsing Action & Action Input dari keluaran teks
        action_match = re.search(r"Action:\s*(.*?)\n", step_text)
        action_input_match = re.search(r"Action Input:\s*(.*?)(?:\n|$)", step_text)

        if action_match and action_input_match:
            action = action_match.group(1).strip()
            action_input = action_input_match.group(1).strip()

            # Jalankan fungsi lokal
            observation = execute_tool(action, action_input)
            print(f"Observation: {observation}")

            # Perbarui scratchpad histori percakapan ReAct
            scratchpad += f"\n{step_text}\nObservation: {observation}"
        else:
            print("[Warning] Format Action tidak terdeteksi, menghentikan loop.")
            break

    return "Investigasi selesai tanpa kesimpulan final."


# ==========================================
# 5. EXECUTION
# ==========================================

def run_demonstration():
    incoming_trigger = (
        "ALERT_SYSTEM: Terdeteksi transaksi penarikan tunai sebesar IDR 200.000.000 "
        "di Singapura untuk nasabah User ID: USR-001 dalam kurun waktu 5 menit terakhir."
    )

    print("\n=== STEP 1: ROUTER RECEIVES TRIGGER ===")
    print(f"Incoming Event: {incoming_trigger}")

    destination = router_node(incoming_trigger)
    print(f"Router Decision: [{destination}]")

    if destination == "AML_AGENT":
        print("\n=== STEP 2: MANUAL REACT AGENT LOOP ===")
        final_result = run_react_agent(incoming_trigger)

        print("\n=== FINAL RESULT ===")
        print(final_result)
    else:
        print("Mengarahkan ke Layanan Pelanggan Umum...")

if __name__ == "__main__":
    run_demonstration()

Memuat processor dan model Gemma lokal...


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== STEP 1: ROUTER RECEIVES TRIGGER ===
Incoming Event: ALERT_SYSTEM: Terdeteksi transaksi penarikan tunai sebesar IDR 200.000.000 di Singapura untuk nasabah User ID: USR-001 dalam kurun waktu 5 menit terakhir.
Router Decision: [GENERAL_QUERY]
Mengarahkan ke Layanan Pelanggan Umum...
